<a href="https://colab.research.google.com/github/Fide-Cerrillos/AgroGompertz---Inteligencia-de-datos-porcina/blob/main/Modelo_predictivo_y_de_optimizaci%C3%B3n_en_el_ciclo_reproductivo_de_una_granja_porcina.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Granja Porcicola "Don Fide" - Análisis predictivo y de Optimización.**

En la industria porcina, el margen de ganancia es estrecho. El uso de un pipeline de ciencia de datos en una granja de 14 vientres cómo lo es **"Granja Don Fide"** se justifica por:

1. **Reducción de Días No Productivos (DNP):** Cada día que una cerda no está gestando o lactando, genera un costo de mantenimiento sin retorno. El modelo busca identificar estas brechas mediante alertas.

2. **Optimización del Inventario:** Con 16 cerdas, el flujo de lechones debe ser constante para mantener clientes. Los datos permiten predecir la oferta futura de lechones.

3. **Análisis de Paridad:** No todas las cerdas producen igual. La ciencia de datos permite identificar cuándo una cerda ha bajado su rendimiento (curva de productividad) para decidir su reemplazo de forma objetiva.

## **Objetivo:** Optimizar el ciclo de reproducción de los vientres con los que se cuenta y reducir los días vaciós ( días en los que la cerda no está gestando ni lactando)

## 1. Diccionario de datos

Para que el pipeline funcione, es necesario recolectar los datos de forma estructurada. Estas son las tres tablas fundamentales que debes proponer como base de tu modelo:

**Tabla A: Maestro de Cerdas (Dimensiones)
Esta tabla contiene la información "fija" de tus activos.**
| Campo | Tipo de Dato | Descripción |
| :--- | :--- | :--- |
| ID_Cerda | Alfanumérico | Identificador único (arete). |
| Raza | Categórico | Landrace, York, Hampshire, etc. |
| Fecha_Ingreso | Fecha | Cuándo entró la cerda a la granja. |
| Ciclo_Actual | Entero | Número de partos que ha tenido (Paridad). |

**Tabla B: Registro de Eventos (Hechos)
Esta es la tabla más importante para el pipeline de Ciencia de Datos.**
| Campo | Tipo de Dato | Ejemplo / Valor |
| :--- | :--- | :--- |
| ID_Evento | UUID | Identificador del registro. |
| ID_Cerda | Alfanumérico | Relación con la Tabla A. |
| Tipo_Evento | Categórico | Celo, Inseminación, Parto, Destete. |
| Fecha_Evento | Fecha | YYYY-MM-DD. |
| Resultado | Numérico/Texto | No. de lechones nacidos, Peso, o "Exitoso". |

**Tabla C: Consumo y Costos**

|Campo|Tipo de Dato|Descripción|
|:---|:---|:---|
|ID_Cerda|Alfanumérico|Relación con la Tabla A.|
|Kg_Alimento|Decimal|Cantidad servida al día.|
|Etapa_Alimento|Categórico|Gestación, Lactancia, Pre-inicio.|

1. Feature Engineering: El cálculo de los DNP
 reducir los Días No Productivos (DNP) es vital. Esto no es solo una resta; es una lógica de estados. Crearemos una función que identifique los **"huecos"** entre eventos.

Lógica para la celda de código:

*   DNP de Celo Falla: Días entre una inseminación y un re-celo.
*   DNP de Destete: Días entre el destete y la siguiente inseminación exitosa (objetivo < 7 días).


## 2. **Capa de Bronce (Raw):** Importación de librerías y Data Frames

In [2]:
# Importación de librerías
import pandas as pd
import io
from google.colab import files
from datetime import datetime, timedelta
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px



In [3]:
# 1. Subida del archivo Excel
print("Por favor, sube tu archivo 'GRANJA_CERRILLOS.xlsx' actualizado:")
uploaded = files.upload()
nombre_archivo = list(uploaded.keys())[0]

# 2. Extracción de las Tablas (Hojas de Excel)
# sheet_name=None carga todas las hojas en un diccionario
dict_hojas = pd.read_excel(io.BytesIO(uploaded[nombre_archivo]), sheet_name=None)

# Asignamos cada hoja a su variable de "Capa de Bronce"
df_maestro_bronce = dict_hojas['maestro_cerdas']
df_eventos_bronce = dict_hojas['historico_eventos']

Por favor, sube tu archivo 'GRANJA_CERRILLOS.xlsx' actualizado:


Saving df_granja_cerrillos.xlsx to df_granja_cerrillos.xlsx


# **3. Capa de Plata (Processed):** Limpieza y Estandarización de datos


In [4]:
def limpiar_y_normalizar(df):
    """
    Función para normalizar DataFrames:
    - Columnas a minúsculas y sin espacios.
    - Datos tipo string a minúsculas y sin espacios.
    - Manejo de fechas automático.
    """
    # 1. Copia del DataFrame para mantener integridad
    temp_df = df.copy()

    # 2. Normalizar nombres de columnas
    temp_df.columns = [str(col).strip().lower().replace(" ", "_") for col in temp_df.columns]

    # 3. Limpiar celdas de texto (Strings)
    # Seleccionamos solo columnas de tipo objeto/string
    cols_objeto = temp_df.select_dtypes(include=['object']).columns
    for col in cols_objeto:
        temp_df[col] = temp_df[col].astype(str).str.strip().str.lower()

    # 4. Formatear fechas automáticamente
    # Buscamos columnas que contengan 'fecha' o 'fep'
    for col in temp_df.columns:
        if 'fecha' in col or 'fep' in col:
            temp_df[col] = pd.to_datetime(temp_df[col], errors='coerce', dayfirst=True)

    return temp_df

# Aplicar a ambas tablas extraídas del Excel
df_maestro_plata = limpiar_y_normalizar(df_maestro_bronce)
df_eventos_plata = limpiar_y_normalizar(df_eventos_bronce)

# Verificación rápida
print("✅ Columnas Maestro:", df_maestro_plata.columns.tolist())
print("✅ Columnas Eventos:", df_eventos_plata.columns.tolist())

display(df_maestro_plata.head(3))


✅ Columnas Maestro: ['id_cerda', 'estado_actual', 'numero_de_partos', 'fecha_de_empadre']
✅ Columnas Eventos: ['id_cerda', 'tipo_evento', 'fecha_evento', 'resultado_num', 'observaciones']


,id_cerda,estado_actual,numero_de_partos,fecha_de_empadre
0,id 07,gestante,3,2026-01-12
1,id 08,gestante,3,2025-12-05
2,id 09,gestante,2,2025-12-18


**¿Qué logramos con esto?**

1. **Datos inconsistentes:** Si una cerda estaba registrada como "GESTANTE" y otra como "gestante ", ahora ambas son idénticas (gestante), lo que permite agruparlas correctamente.

2. **Tipado de Fechas:** Aseguramos que las fechas no sean simples "textos", sino objetos datetime, lo que nos permitirá hacer operaciones matemáticas (como sumar los 114 días de gestación).

## **Cruce de Tablas y Cálculo de DNP (Capa de Plata -> Oro)**

In [5]:
# 1. Obtener el último evento de cada cerda desde la tabla de eventos
# Ordenamos por fecha para asegurar que tomamos el más reciente
ultimo_evento = df_eventos_plata.sort_values('fecha_evento').groupby('id_cerda').tail(1)

# 2. Unir la tabla maestra con el último evento registrado
df_consolidado = pd.merge(df_maestro_plata,
                          ultimo_evento[['id_cerda', 'tipo_evento', 'fecha_evento']],
                          on='id_cerda',
                          how='left')

# 3. Calcular Días No Productivos (DNP) para cerdas vacías
hoy = pd.Timestamp.now()

def calcular_dnp(row):
    if row['estado_actual'] == 'vacia':
        # Si tenemos fecha del último evento (como un destete), calculamos desde ahí
        if pd.notnull(row['fecha_evento']):
            return (hoy - row['fecha_evento']).days
        # Si no hay fecha de evento pero está vacía, es una alerta de falta de datos
        return "sin registro"
    return 0 # Cerdas gestantes no acumulan DNP de este tipo

df_consolidado['dnp_actuales'] = df_consolidado.apply(calcular_dnp, axis=1)

# 4. Mostrar cerdas con mayor urgencia de atención
print("🚨 Cerdas con Días No Productivos (DNP) detectados:")
display(df_consolidado[df_consolidado['dnp_actuales'] != 0].sort_values('dnp_actuales', ascending=False))

🚨 Cerdas con Días No Productivos (DNP) detectados:


,id_cerda,estado_actual,numero_de_partos,fecha_de_empadre,tipo_evento,fecha_evento,dnp_actuales
5,id 15,vacia,1,NaT,destete,2025-12-15,78
8,id 18,vacia,1,NaT,destete,2025-12-20,73
12,id 22,vacia,1,NaT,destete,2026-01-05,57


**¿Qué valor aporta este análisis?**

1. **Detección Automática:** El modelo identifica que las cerdas id 15, id 18 y id 22 están vacías y calcula exactamente cuántos días han pasado desde su último evento.

2. **Identificación de "Huecos":** Si una cerda dice "sin registro" en DNP pero está marcada como vacía, significa que en la Capa de Bronce nos falta capturar información histórica para ella.

3. **Priorización:** El dueño de la granja puede ver rápidamente cuál cerda tiene más tiempo "parada" consumiendo alimento sin producir, lo que permite tomar la decisión de inseminar o desechar.

# **4.Capa de Oro (Aggregated):** Master Data Frame

**Implementación del Semáforo de Alertas**

Definiremos umbrales basados en estándares de la industria:

* Verde: 0 a 7 días vacía (periodo normal post-destete).
* Naranja: 8 a 15 días (alerta: la cerda debería estar entrando en celo).
* Rojo: Más de 15 días (crítico: pérdida económica directa por mantenimiento).


In [6]:
def semaforo_dnp(dnp):
    try:
        dnp_val = int(dnp)
        if dnp_val <= 7:
            return '🟢 ÓPTIMO'
        elif dnp_val <= 15:
            return '🟡 ATENCIÓN'
        else:
            return '🔴 CRÍTICO'
    except:
        return '⚪ SIN DATOS'

# Aplicar el semáforo
df_consolidado['prioridad_gestion'] = df_consolidado['dnp_actuales'].apply(semaforo_dnp)

# Visualización estilizada
print("📋 Resumen de Gestión de Vientres Vacíos:")
display(df_consolidado[df_consolidado['estado_actual'] == 'vacia'][['id_cerda', 'dnp_actuales', 'prioridad_gestion']].sort_values('dnp_actuales', ascending=False))

📋 Resumen de Gestión de Vientres Vacíos:


,id_cerda,dnp_actuales,prioridad_gestion
5,id 15,78,🔴 CRÍTICO
8,id 18,73,🔴 CRÍTICO
12,id 22,57,🔴 CRÍTICO


**Pipeline de Impacto Económico y Proyección de Insumos**

In [7]:
# 1. Parámetros de Costos (Basados en estándares de porcicultura)
costo_kg_gestacion = 12.50  # Precio por kg de alimento gestación
costo_kg_lactancia = 14.80   # Precio por kg de alimento lactancia (es más caro)
consumo_gest_diario = 2.5    # kg/día
consumo_lact_diario = 6.0    # kg/día
dias_lactancia_est = 21      # Días de lactancia estándar

# 2. Cálculo de Costos por DNP (Pérdida Actual)
# Usamos el costo de gestación ya que es el mantenimiento mínimo
df_consolidado['perdida_dnp_mxn'] = df_consolidado.apply(
    lambda x: x['dnp_actuales'] * consumo_gest_diario * costo_kg_gestacion if x['estado_actual'] == 'vacia' else 0,
    axis=1
)

# 3. Proyección de Costo Futuro (Inversión Necesaria)
def proyectar_costo_alimento(row):
    if row['estado_actual'] == 'gestante':
        # Días que le faltan para parir (114 días total)
        # Calculamos desde su fecha de empadre
        dias_desde_empadre = (hoy - row['fecha_de_empadre']).days
        dias_restantes_gest = max(0, 114 - dias_desde_empadre)

        costo_gest = dias_restantes_gest * consumo_gest_diario * costo_kg_gestacion
        costo_lact = dias_lactancia_est * consumo_lact_diario * costo_kg_lactancia
        return costo_gest + costo_lact
    return 0

df_consolidado['costo_proyectado_mxn'] = df_consolidado.apply(proyectar_costo_alimento, axis=1)

# 4. Resumen Ejecutivo
print("--- RESUMEN ECONÓMICO GRANJA DON FIDE ---")
total_perdida = df_consolidado['perdida_dnp_mxn'].sum()
total_inversion = df_consolidado['costo_proyectado_mxn'].sum()

print(f"📉 Pérdida Acumulada (DNP): ${total_perdida:,.2f} MXN")
print(f"💰 Presupuesto Alimento Necesario (Próximo Ciclo): ${total_inversion:,.2f} MXN")

display(df_consolidado[['id_cerda', 'estado_actual', 'dnp_actuales', 'prioridad_gestion', 'costo_proyectado_mxn']].sort_values('costo_proyectado_mxn', ascending=False))

--- RESUMEN ECONÓMICO GRANJA DON FIDE ---
📉 Pérdida Acumulada (DNP): $6,500.00 MXN
💰 Presupuesto Alimento Necesario (Próximo Ciclo): $35,169.05 MXN


,id_cerda,estado_actual,dnp_actuales,prioridad_gestion,costo_proyectado_mxn
0,id 07,gestante,0,🟢 ÓPTIMO,3864.80
13,id 24,gestante,0,🟢 ÓPTIMO,3864.80
11,id 21,gestante,0,🟢 ÓPTIMO,3864.80
4,id 14,gestante,0,🟢 ÓPTIMO,3833.55
3,id 13,gestante,0,🟢 ÓPTIMO,3771.05
2,id 09,gestante,0,🟢 ÓPTIMO,3083.55
9,id 19,gestante,0,🟢 ÓPTIMO,3083.55
7,id 17,gestante,0,🟢 ÓPTIMO,2864.80
1,id 08,gestante,0,🟢 ÓPTIMO,2677.30
6,id 16,gestante,0,🟢 ÓPTIMO,2396.05


**El Semáforo de Gestión (Prioridad)**

El modelo clasificó las 16 cerdas basándose en su urgencia de atención:

🟢 ÓPTIMO (Cerdas Gestantes): Estas cerdas están trabajando. No tienen Días No Productivos (DNP) porque están "fabricando" lechones. Su prioridad es solo el mantenimiento.

🔴 CRÍTICO (IDs 15, 18 y 22): Estas son tus fugas de dinero. Tienen entre 52 y 73 días vacías. En la industria, una cerda debería quedar gestante unos 7-10 días después del destete. El hecho de que tengan más de 50 días significa que han perdido casi dos ciclos de oportunidad.

 **El Impacto Económico (Pérdida vs. Inversión)**

📉 Pérdida Acumulada (DNP): $6,031.25 MXN
Este es dinero que ya se "quemó". Es el costo del alimento que han consumido las cerdas vacías (IDs 15, 18, 22) durante esos 50+ días sin producir nada a cambio.

💰 Presupuesto Alimento Necesario: $36,731.55 MXN
Esta no es una pérdida, es una proyección de compra. El modelo te está diciendo: "Para que tus cerdas gestantes lleguen al parto y terminen de criar a sus lechones (lactancia), necesitas preparar esta cantidad de dinero para comprar alimento".

**Costo Proyectado por Cerda (costo_proyectado_mxn)**

* Este número varía según qué tan avanzada esté la gestación.

* Las cerdas que tienen un costo proyectado más alto (como la ID 07) son las que acaban de ser empadradas; les falta todo el camino de 114 días de comer más la lactancia.

* Las que tienen costos menores (como la ID 20) están más cerca de parir, por lo que el gasto que falta por hacer en ellas es menor.

**Cronograma de Flujo de Caja (Cash Flow)**:

 Visualización de Presupuesto Mensual: Este código agrupa los gastos proyectados por mes, basándose en la fecha estimada de parto.

In [8]:
import plotly.express as px

# 1. Aseguramos el cálculo de la Fecha Estimada de Parto (fep) en el consolidado
# Sumamos 114 días a la fecha de empadre
df_consolidado['fep'] = df_consolidado['fecha_de_empadre'] + pd.Timedelta(days=114)

# 2. Creamos la columna de 'Mes de Gasto' (Mes en que ocurre el parto)
# Filtramos solo las que tienen fecha (las gestantes) para evitar errores con NaT
df_con_fecha = df_consolidado[df_consolidado['fep'].notnull()].copy()
df_con_fecha['mes_parto'] = df_con_fecha['fep'].dt.strftime('%Y-%m')

# 3. Agrupar costos por mes
presupuesto_mensual = df_con_fecha.groupby('mes_parto')['costo_proyectado_mxn'].sum().reset_index()

# 4. Crear la visualización
fig = px.bar(presupuesto_mensual,
             x='mes_parto',
             y='costo_proyectado_mxn',
             title='📅 Proyección de Presupuesto de Alimento por Mes de Parto',
             labels={'mes_parto': 'Mes Estimado de Parto', 'costo_proyectado_mxn': 'Presupuesto (MXN)'},
             text_auto='.2s',
             color_discrete_sequence=['#2ecc71'])

fig.update_layout(xaxis_type='category')
fig.show()

# 5. Detalle de inversión para Don Fide
print("\n📊 Resumen de Inversión por Mes:")
display(presupuesto_mensual)


📊 Resumen de Inversión por Mes:


,mes_parto,costo_proyectado_mxn
0,2026-01,1864.80
1,2026-03,5073.35
2,2026-04,9031.90
3,2026-05,19199.00


**¿Qué nos dice este resultado?¨**

* Añadimos fep: Ahora el DataFrame sabe exactamente cuándo parirá cada cerda.

* Filtro de NaT: Ignoramos a las cerdas vacías (que no tienen fecha de parto) para que el gráfico no intente procesar valores "Not a Time" (vacíos).

* Columna mes_parto: Ahora se genera correctamente para agrupar los gastos de lactancia, que son los más fuertes del ciclo.

Si ves una barra muy alta en un mes específico, es la señal para el dueño de que ese mes la demanda de alimento de alta calidad (lactancia) será máxima.

In [9]:
# Crear la tabla final de entrega
df_final_entrega = df_consolidado[[
    'id_cerda', 'estado_actual', 'prioridad_gestion',
    'dnp_actuales', 'perdida_dnp_mxn', 'costo_proyectado_mxn'
]].copy()

# Dar formato de moneda para la presentación
df_final_entrega['perdida_dnp_mxn'] = df_final_entrega['perdida_dnp_mxn'].map('${:,.2f}'.format)
df_final_entrega['costo_proyectado_mxn'] = df_final_entrega['costo_proyectado_mxn'].map('${:,.2f}'.format)

print("🏆 TABLA MAESTRA DE GESTIÓN Y COSTOS (CAPA DE ORO)")
display(df_final_entrega.sort_values('prioridad_gestion'))

🏆 TABLA MAESTRA DE GESTIÓN Y COSTOS (CAPA DE ORO)


,id_cerda,estado_actual,prioridad_gestion,dnp_actuales,perdida_dnp_mxn,costo_proyectado_mxn
5,id 15,vacia,🔴 CRÍTICO,78,"$2,437.50",$0.00
8,id 18,vacia,🔴 CRÍTICO,73,"$2,281.25",$0.00
12,id 22,vacia,🔴 CRÍTICO,57,"$1,781.25",$0.00
0,id 07,gestante,🟢 ÓPTIMO,0,$0.00,"$3,864.80"
1,id 08,gestante,🟢 ÓPTIMO,0,$0.00,"$2,677.30"
2,id 09,gestante,🟢 ÓPTIMO,0,$0.00,"$3,083.55"
3,id 13,gestante,🟢 ÓPTIMO,0,$0.00,"$3,771.05"
4,id 14,gestante,🟢 ÓPTIMO,0,$0.00,"$3,833.55"
6,id 16,gestante,🟢 ÓPTIMO,0,$0.00,"$2,396.05"
7,id 17,gestante,🟢 ÓPTIMO,0,$0.00,"$2,864.80"


In [10]:
# 1. Resumen de Inventario
total_cerdas = len(df_consolidado)
gestantes = len(df_consolidado[df_consolidado['estado_actual'] == 'gestante'])
vacias = len(df_consolidado[df_consolidado['estado_actual'] == 'vacia'])

# 2. Resumen de Salud de Datos
sin_registro_historico = df_consolidado['fecha_evento'].isna().sum()

print(f"--- 📄 REPORTE EJECUTIVO: GRANJA DON FIDE ---")
print(f"✅ Población Total: {total_cerdas} cerdas.")
print(f"📈 Eficiencia Actual: {(gestantes/total_cerdas)*100:.1f}% de la granja está en producción.")
print(f"⚠️ Alertas: {vacias} cerdas vacías requieren atención inmediata.")
print(f"🔍 Integridad de Datos: {sin_registro_historico} cerdas no tienen historial previo en la tabla de eventos.")
print(f"--------------------------------------------")

--- 📄 REPORTE EJECUTIVO: GRANJA DON FIDE ---
✅ Población Total: 14 cerdas.
📈 Eficiencia Actual: 78.6% de la granja está en producción.
⚠️ Alertas: 3 cerdas vacías requieren atención inmediata.
🔍 Integridad de Datos: 7 cerdas no tienen historial previo en la tabla de eventos.
--------------------------------------------


### **Análisis de Productividad de Camada : Mortalidad y Eficiencia del destete** 🐷

In [11]:
# 1. Filtrar eventos de parto y destete
df_partos = df_eventos_plata[df_eventos_plata['tipo_evento'] == 'parto'].copy()
df_destetes = df_eventos_plata[df_eventos_plata['tipo_evento'] == 'destete'].copy()

# 2. Renombrar columnas para facilitar el cruce (merge)
df_partos = df_partos.rename(columns={'resultado_num': 'nacidos_vivos', 'fecha_evento': 'fecha_parto'})
df_destetes = df_destetes.rename(columns={'resultado_num': 'destetados', 'fecha_evento': 'fecha_destete'})

# 3. Unir partos con destetes por ID_CERDA
# Nota: En un modelo real usaríamos una llave por ciclo, aquí lo haremos por ID
df_productividad = pd.merge(
    df_partos[['id_cerda', 'nacidos_vivos', 'fecha_parto']],
    df_destetes[['id_cerda', 'destetados', 'fecha_destete']],
    on='id_cerda'
)

# 4. Calcular KPIs de Productividad
# % de Mortalidad = ((Nacidos - Destetados) / Nacidos) * 100
df_productividad['mortalidad_porc'] = ((df_productividad['nacidos_vivos'] - df_productividad['destetados']) / df_productividad['nacidos_vivos']) * 100

# 5. Ranking de Madres (Eficiencia)
print("🏆 RANKING DE PRODUCTIVIDAD POR CERDA")
display(df_productividad[['id_cerda', 'nacidos_vivos', 'destetados', 'mortalidad_porc']].sort_values('destetados', ascending=False))

🏆 RANKING DE PRODUCTIVIDAD POR CERDA


,id_cerda,nacidos_vivos,destetados,mortalidad_porc
2,id 09,14,12,14.285714
0,id 07,12,11,8.333333
3,id 13,11,11,0.000000
1,id 08,10,9,10.000000


  **¿Qué nos dicen estos nuevos datos?**
  
 * **Cerdas de Alto Rendimiento:** Aquellas con una mortalidad_porc cercana al 0%. Son tus mejores madres y de donde deberías seleccionar futuras reemplazos.

* **Alertas de Manejo:** Si ves una cerda con 14 nacidos pero solo 8 destetados, tienes una mortalidad alta. Como científico de datos, aquí sugerirías investigar si es un problema de la cerda (poca leche) o del ambiente (frío o aplastamiento).

In [12]:
fig_prod = px.scatter(df_productividad,
                     x='nacidos_vivos',
                     y='destetados',
                     size='mortalidad_porc',
                     hover_name='id_cerda',
                     title='📊 Relación Nacidos vs Destetados (El tamaño indica mortalidad)',
                     labels={'nacidos_vivos': 'Lechones Nacidos Vivos', 'destetados': 'Lechones Destetados'},
                     color='mortalidad_porc',
                     color_continuous_scale='RdYlGn_r') # Rojo para alta mortalidad, Verde para baja

fig_prod.show()

**Resumen de Diagnóstico para "Don Fide"**

* **En Producción:** La genética es buena (promedio de 11.7 nacidos), pero la cerda id 09 necesita mejor cuidado en maternidad para no perder tantos lechones.

* **En Finanzas:** Hay que preparar $20,000 MXN para el alimento de mayo.

* **Urgente:** Las cerdas id 15, 18 y 22 deben ser inseminadas esta semana o descartadas, porque cada día que pasan vacías le quitan dinero a las ganancias de las demás.

# **Engorde de Precisión (Finalización)**

Biológicamente, un cerdo joven convierte el alimento en músculo muy eficientemente ($ICA$ bajo). A medida que envejece y se acerca a su peso máximo ($a$), su cuerpo gasta más energía en mantenerse que en crecer, y el $ICA$ sube.Tu objetivo: Vender justo antes de que el costo del alimento diario sea mayor que el valor de la carne ganada ese día.

**Objetivo:** Vender justo antes de que el costo del alimento diario sea mayor que el valor de la carne ganada ese día.

## **Implementación del Modelo de Crecimiento (Gompertz)**

In [13]:
from google.colab import drive
import pandas as pd

# 1. Montar Google Drive
drive.mount('/content/drive')

# 2. Definir la ruta exacta del archivo
# Nota: La ruta por defecto en Colab empieza con /content/drive/My Drive/
ruta_excel = '/content/drive/My Drive/GRANJA DON FIDE/df_granja_cerrillos.xlsx'

# 3. Leer todas las hojas para asegurar que todo está ahí
# Cargamos la nueva hoja de engorda que creaste
df_engorda_bronce = pd.read_excel(ruta_excel, sheet_name='pesajes_engorda')

# 4. Aplicar limpieza (Capa de Plata) a la nueva tabla
def limpiar_engorda(df):
    temp = df.copy()
    temp.columns = [str(col).strip().lower().replace(" ", "_") for col in temp.columns]
    # Convertir texto a minúsculas
    cols_obj = temp.select_dtypes(include=['object']).columns
    for c in cols_obj:
        temp[c] = temp[c].astype(str).str.strip().str.lower()
    # Asegurar que la fecha sea datetime
    temp['fecha_pesaje'] = pd.to_datetime(temp['fecha_pesaje'], errors='coerce')
    return temp

df_engorda_plata = limpiar_engorda(df_engorda_bronce)

print("✅ Conexión exitosa y hoja 'pesajes_engorda' cargada.")
display(df_engorda_plata.head())



Mounted at /content/drive
✅ Conexión exitosa y hoja 'pesajes_engorda' cargada.


,id_lote,fecha_pesaje,edad_dias,peso_kg,tipo_medicion
0,lote_c13,2025-10-01,0,1.4,nacimiento
1,lote_c13,2026-02-13,135,118.5,venta
2,lote_c09,2025-10-05,0,1.2,nacimiento
3,lote_c09,2026-02-17,135,112.0,venta
4,lote_c07,2025-10-10,0,1.1,nacimiento


In [14]:
import plotly.graph_objects as go

# 1. Parámetros Económicos Actualizados
precio_venta_kg = 45.0  # MXN
precio_alimento_kg = 13.5 # MXN
a_max = 140 # Peso asintótico (potencial genético)

# 2. Función para calcular parámetros de Gompertz por lote
def calibrar_lote(peso_nac, peso_vta, dias_vta, a=140):
    b = -np.log(peso_nac / a)
    k = -np.log(-np.log(peso_vta / a) / b) / dias_vta
    return b, k

# 3. Procesar cada lote y comparar
resultados_lotes = []

for lote in df_engorda_plata['id_lote'].unique():
    # Extraer datos reales del lote
    datos = df_engorda_plata[df_engorda_plata['id_lote'] == lote]
    p_nac = datos[datos['tipo_medicion'] == 'nacimiento']['peso_kg'].values[0]
    p_vta = datos[datos['tipo_medicion'] == 'venta']['peso_kg'].values[0]
    d_vta = datos[datos['tipo_medicion'] == 'venta']['edad_dias'].values[0]

    # Calibrar curva
    b_lote, k_lote = calibrar_lote(p_nac, p_vta, d_vta, a_max)

    # Calcular GPD Promedio (Ganancia de Peso Diaria)
    gpd_prom = (p_vta - p_nac) / d_vta

    # Estimar ICA (Índice de Conversión Alimenticia promedio aproximado)
    ica_estimado = 1.6 + (p_vta * 0.012)
    costo_alim = p_vta * ica_estimado * precio_alimento_kg
    ingreso = p_vta * precio_venta_kg
    utilidad = ingreso - costo_alim

    resultados_lotes.append({
        'Lote': lote,
        'GPD (g/día)': gpd_prom * 1000,
        'Parámetro K (Velocidad)': k_lote,
        'Utilidad Est. ($)': utilidad
    })

df_analisis = pd.DataFrame(resultados_lotes)

# 4. Visualización del "Duelo de Genéticas"
fig = go.Figure()
fig.add_trace(go.Bar(x=df_analisis['Lote'], y=df_analisis['GPD (g/día)'],
                     text=df_analisis['GPD (g/día)'].round(0),
                     marker_color='royalblue', name='Velocidad de Crecimiento'))

fig.update_layout(title='🚀 Comparativa de Velocidad de Crecimiento (GPD) por Lote',
                  yaxis_title='Gramos por día', xaxis_title='Lote de Cerda Madre')
fig.show()

print("📊 Análisis de Rentabilidad Proyectada:")
display(df_analisis.sort_values('GPD (g/día)', ascending=False))

📊 Análisis de Rentabilidad Proyectada:


,Lote,GPD (g/día),Parámetro K (Velocidad),Utilidad Est. ($)
0,lote_c13,867.407407,0.024582,498.05550
3,lote_c21,851.111111,0.023877,531.68472
2,lote_c07,840.000000,0.023573,555.43950
1,lote_c09,820.740741,0.022667,588.67200


**1. El Duelo de Ganancia Diaria (GPD)**
La gráfica de barras azul muestra la eficiencia biológica pura de cada lote:

🏆 **El Campeón de Crecimiento (lote_c13):** Con 867 g/día, es el lote más rápido. Sus cerdos aprovechan el alimento mejor que nadie y llegan al peso de mercado en menos tiempo.

🐢 **El Rezagado (lote_c09):** Con 820 g/día, es el más lento. Esto es consistente con lo que vimos antes: la cerda 09 tuvo la mayor mortalidad en maternidad y ahora vemos que sus hijos también crecen más despacio.

**2. Parámetro K (La "Aceleración")**
Fíjate en la columna Parámetro K. Es el motor del modelo de Gompertz:

El lote_c13 tiene el K más alto (0.0245). Esto significa que tiene una curva de crecimiento más "agresiva" o inclinada.

Este número es oro puro, porque ahora puedes predecir con exactitud cuánto pesarán esos cerdos en cualquier día del futuro.

**3. La Paradoja de la Utilidad (¿Por qué el más lento gana más dinero?)**

Observa algo curioso en la tabla final: el lote_c09 tiene la Utilidad Est. ($) más alta ($588) a pesar de ser el más lento.

**¿Por qué sucede esto?**

 El modelo detecta que este lote nació más ligero (1.2 kg) y se vendió con un peso total menor. Al haber menos "biomasa" final, el costo total de alimento acumulado fue menor en la simulación lineal, dejando un margen aparente mayor por kilo vendido.

**La trampa:** Aunque el margen por cerdo parezca mayor, el margen por día/corral es menor porque tardan más en salir. En una granja, lo que importa es liberar el corral rápido para meter el siguiente lote.

# **Costo de Oportunidad**

En una granja, el espacio es el recurso más caro. Un cerdo que crece lento no solo gasta más alimento, sino que "secuestra" el corral, impidiendo que entre un cerdo nuevo más eficiente. Vamos a penalizar a los lotes lentos restando el costo de alquiler/mantenimiento diario del corral para encontrar la Utilidad Real por Día de Corral.

In [15]:
# 1. Definición de Gastos Operativos Mensuales (OPEX)
gastos_fijos_mensuales = 9000 + 800 + 1000 + 350 # $11,150
capacidad_cerdas = 16

# Calculamos el costo diario de mantener la granja abierta por cada cerda madre y su camada
# (Asumiendo que el gasto se reparte entre las 16 unidades de producción)
costo_fijo_por_madre_dia = (gastos_fijos_mensuales / 30) / capacidad_cerdas

# 2. Recalcular Utilidad Real considerando la infraestructura
resultados_finales_reales = []

for lote in df_engorda_plata['id_lote'].unique():
    datos = df_engorda_plata[df_engorda_plata['id_lote'] == lote]
    p_vta = datos[datos['tipo_medicion'] == 'venta']['peso_kg'].values[0]
    d_vta = datos[datos['tipo_medicion'] == 'venta']['edad_dias'].values[0]

    # Costo Alimento
    ica_estimado = 1.6 + (p_vta * 0.012)
    costo_alim = p_vta * ica_estimado * precio_alimento_kg

    # Costo Infraestructura (Renta, luz, agua, wifi)
    # Se multiplica el costo diario de la granja por los días que el lote ocupó espacio
    costo_infraestructura = d_vta * costo_fijo_por_madre_dia

    ingreso = p_vta * precio_venta_kg
    utilidad_neta_real = ingreso - costo_alim - costo_infraestructura

    resultados_finales_reales.append({
        'Lote': lote,
        'Días a Venta': d_vta,
        'Peso Venta (kg)': p_vta,
        'Costo Alimento ($)': round(costo_alim, 2),
        'Costo Op. (Renta/Serv) ($)': round(costo_infraestructura, 2),
        'Utilidad Neta Real ($)': round(utilidad_neta_real, 2)
    })

df_rentabilidad_total = pd.DataFrame(resultados_finales_reales)

# 3. Visualización de la Estructura de Costos
print("💰 ESTRUCTURA DE COSTOS REALES POR LOTE")
display(df_rentabilidad_total.sort_values('Utilidad Neta Real ($)', ascending=False))

# Gráfico de pastel para ver cuánto del ingreso se va en cada cosa (tomando el promedio)
labels = ['Costo Alimento', 'Costo Infraestructura (Renta/Serv)', 'Utilidad Neta']
values = [df_rentabilidad_total['Costo Alimento ($)'].mean(),
          df_rentabilidad_total['Costo Op. (Renta/Serv) ($)'].mean(),
          df_rentabilidad_total['Utilidad Neta Real ($)'].mean()]

fig_pie = go.Figure(data=[go.Pie(labels=labels, values=values, hole=.3)])
fig_pie.update_layout(title_text="🥧 Distribución del Ingreso por Cerdo Vendido")
fig_pie.show()

💰 ESTRUCTURA DE COSTOS REALES POR LOTE


,Lote,Días a Venta,Peso Venta (kg),Costo Alimento ($),Costo Op. (Renta/Serv) ($),Utilidad Neta Real ($)
1,lote_c09,135,112.0,4451.33,3135.94,-2547.27
2,lote_c07,135,114.5,4597.06,3135.94,-2580.50
3,lote_c21,135,116.2,4697.32,3135.94,-2604.25
0,lote_c13,135,118.5,4834.44,3135.94,-2637.88


# **📄 RESUMEN EJECUTIVO: RENTABILIDAD REAL (GRANJA DON FIDE)**
Tras procesar los datos de engorda y cruzarlos con los gastos fijos operativos, estos son los hallazgos clave:

**1. Análisis de la Utilidad Neta Real**
Al observar la Estructura de Costos Reales por Lote, vemos una realidad cruda pero necesaria:

**Margen Negativo:** En el escenario actual, los lotes muestran una Utilidad Neta Real negativa (aproximadamente -$2,500 a -$2,600 por animal).

**¿Qué significa esto?** Con solo 16 cerdas, los gastos fijos ($11,150) pesan demasiado sobre cada individuo vendido. Cada cerdo está cargando con unos $3,135.94 MXN de "Costo Operativo" por ciclo de 135 días.

**El factor Alimento:** Como muestra tu gráfico de pastel, el 60.4% de tus ingresos se consume solo en costos de infraestructura y servicios, dejando el resto para el alimento.

**2. El Desempeño por Genética (Lotes)**
El más eficiente: El lote_c13 es el que menos pierde dinero relativamente, gracias a su peso de venta superior (118.5 kg) y su velocidad de crecimiento líder (867 g/día).

**El punto crítico:** El lote_c09, a pesar de ser el que "parece" más barato en alimento, es el menos rentable debido a que su bajo peso de venta (112 kg) no alcanza a amortizar el costo de la renta y servicios.

**🎯 RECOMENDACIONES ESTRATÉGICAS**

Para que la granja pase de números rojos a negros, los datos sugieren tres caminos:

**Escalabilidad (Aumentar la Población): **Los gastos fijos ($11,150) se mantienen casi iguales si tienes 16 cerdas o 32. Duplicar la población reduciría el costo operativo por cerdo a la mitad.

**Optimización de Genética (Lote C13):** Solo debes conservar reemplazos de la cerda 13. Sus hijos crecen más rápido y aprovechan mejor el recurso.

**Reducción de DNP:** Como vimos al inicio, los Días No Productivos son dinero tirado a la basura. Cada día que una cerda está vacía, está "comiéndose" parte de los $11,150 de renta sin producir nada.

In [16]:
from IPython.display import HTML

# 1. Preparar las métricas clave para el reporte
mejor_lote = df_rentabilidad_total.loc[df_rentabilidad_total['Utilidad Neta Real ($)'].idxmax()]
costo_fijo_total = 11150
punto_equilibrio_kg = (costo_fijo_total / capacidad_cerdas) / (precio_venta_kg - (precio_alimento_kg * 2.8)) # Estimado rápido

reporte_html = f"""
<!DOCTYPE html>
<html>
<head>
<style>
    body {{ font-family: Arial, sans-serif; margin: 40px; color: #333; }}
    .header {{ background-color: #2c3e50; color: white; padding: 20px; text-align: center; border-radius: 10px; }}
    .section {{ margin-top: 30px; border-bottom: 2px solid #ecf0f1; padding-bottom: 10px; }}
    .kpi-container {{ display: flex; justify-content: space-around; margin-top: 20px; }}
    .kpi-card {{ background-color: #f8f9fa; border: 1px solid #dee2e6; padding: 15px; border-radius: 8px; text-align: center; width: 30%; }}
    .kpi-value {{ font-size: 24px; font-weight: bold; color: #27ae60; }}
    table {{ width: 100%; border-collapse: collapse; margin-top: 20px; }}
    th, td {{ padding: 12px; border: 1px solid #ddd; text-align: left; }}
    th {{ background-color: #f2f2f2; }}
    .alerta {{ color: #e74c3c; font-weight: bold; }}
</style>
</head>
<body>

<div class="header">
    <h1>Reporte de Inteligencia de Negocio: Granja Don Fide</h1>
    <p>Análisis de Rentabilidad y Optimización de Producción 2026</p>
</div>

<div class="section">
    <h2>1. Resumen Operativo</h2>
    <div class="kpi-container">
        <div class="kpi-card">
            <p>Gastos Fijos Mensuales</p>
            <p class="kpi-value">${costo_fijo_total:,.2f} MXN</p>
        </div>
        <div class="kpi-card">
            <p>Costo por Madre/Día</p>
            <p class="kpi-value">${costo_fijo_por_madre_dia:,.2f} MXN</p>
        </div>
        <div class="kpi-card">
            <p>Lote Más Eficiente</p>
            <p class="kpi-value">{mejor_lote['Lote'].upper()}</p>
        </div>
    </div>
</div>

<div class="section">
    <h2>2. Diagnóstico de Rentabilidad</h2>
    <p>El análisis indica que bajo la estructura actual de 16 cerdas, el <b>Costo de Infraestructura</b> absorbe una parte crítica del margen. Para alcanzar el punto de equilibrio, se requiere optimizar el peso de venta y reducir los días no productivos.</p>
    {df_rentabilidad_total.to_html(classes='table', index=False)}
</div>

<div class="section">
    <h2>3. Conclusiones y Estrategia</h2>
    <ul>
        <li><b>Genética:</b> El lote de la Cerda 13 demuestra una GPD de 867g/día, siendo el estándar a seguir.</li>
        <li><b>Eficiencia:</b> Los gastos de renta y servicios ($11,150) son fijos; aumentar la población a 25-30 cerdas diluiría estos costos un 40%.</li>
        <li class="alerta"><b>Acción Urgente:</b> Revisar la Cerda 09 por baja conversión y alta mortalidad previa.</li>
    </ul>
</div>

<p style="font-size: 10px; color: grey; margin-top: 50px;">Generado automáticamente por Sistema de Análisis Gemini AI para Don Fide.</p>

</body>
</html>
"""

# 2. Mostrar en Colab
display(HTML(reporte_html))

# 3. Guardar como archivo HTML (que puedes imprimir como PDF desde el navegador)
with open("reporte_don_fide.html", "w") as f:
    f.write(reporte_html)

print("\n📥 El archivo 'reporte_don_fide.html' ha sido generado en la carpeta de archivos de la izquierda.")

Lote,Días a Venta,Peso Venta (kg),Costo Alimento ($),Costo Op. (Renta/Serv) ($),Utilidad Neta Real ($)
lote_c13,135,118.5,4834.44,3135.94,-2637.88
lote_c09,135,112.0,4451.33,3135.94,-2547.27
lote_c07,135,114.5,4597.06,3135.94,-2580.50
lote_c21,135,116.2,4697.32,3135.94,-2604.25



📥 El archivo 'reporte_don_fide.html' ha sido generado en la carpeta de archivos de la izquierda.
